In [1]:
# some needed tools
from typing import Optional
from IPython.display import Markdown, display
from pathlib import Path
from shutil import rmtree
from inu.utils import logs

logs.logger('resman', level='CRITICAL')

<Logger resman (CRITICAL)>

# Resource Management

In [2]:
# markdown("concepts.md")

Resources are _typed_ data items with attributes structure defined by their type (pydantic model).

Resource management system consists of 3 layers:

1. resources - multiple of every resource type (_model_)
2. resource managers - _one_ per every resource _model_ class
3. models manager - _one_ for all the _resource managers_

`ModelsManager` provides _class methods_ to manage instances of `ResourceManager` for every resource type. 

In [3]:
from inu.resman import ModelsManager, ResourceModel, locatable
mm = ModelsManager()    # ModelManager can be used as class or as instance
mm._reset()            # reset in case the notebook is rerun multiple times
print("Found resource managers:", mm.list())

Found resource managers: []


### Basic Resource Management 
Let's start from low level straightforward resource management scenarios.  

#### Creatining New Resource Types
Resource managers are registered automatically each time a new `ResourceModel` sub-class is deinfed.   
`ResourceModel` introduces two mandatory attributes (fields): 
   - name: mandatory string uniquely identifying each resource instance
   - cfg_file_info: - structure describing resource's `.yml` config file, if it was constructed from it.

In [4]:
class ResAModel(ResourceModel):
    description: str = "Resource of type A"

class ResBModel(ResourceModel):
    description: str = "Resource of type B"    

print(f"Found resource managers:", *mm.list(), sep='\n ')

Found resource managers:
 <ResManager:ResA>[0] scan(FIRST:0) in Locator<EIA> []
 <ResManager:ResB>[0] scan(FIRST:0) in Locator<EIA> []


Currently all the resource managers count `[0]` of associated resources.  
Lets create a resource and add it to the its manager:

In [5]:
ra1 = ResAModel('a1', description='first resource')   # a new resource - not registed by its manager!
rmA = ModelsManager.find('A')                   # also ModelsManager.find(ResAModel)
if rmA:
    rmA.add_resource(ra1, over=True)                # allow to overwrite if resource with this name is already registered
rmA                                             # now resource counter is [1]!

<ResManager:ResA>[1] scan(FIRST:0) in Locator<EIA> []

**Note**
 - Every `ResourceModel` class has access to its `ResourceManager` _instance_ through `_manager` attribute.  
 - Every `ResourceManager` _instance_ has access to its `ResourceModel` _class_ through `model` attribute.

In [6]:
(rmB:= ResBModel._manager).add_resource(rmB.model('b1'), over=True)
rmB

<ResManager:ResB>[1] scan(FIRST:0) in Locator<EIA> []

In [7]:
print(f"Found resource managers:", *mm.list(), sep='\n ')

Found resource managers:
 <ResManager:ResA>[1] scan(FIRST:0) in Locator<EIA> []
 <ResManager:ResB>[1] scan(FIRST:0) in Locator<EIA> []
 <ResManager:DataSourceRM>[0] scan(FIRST:0) in Locator<EIA>$INU_DATASETS=/home/ilyap/data/depth []
 <ResManager:SchemeRM>[0] scan(FIRST:0) in Locator<EIA|'/schemes'> [/home/ilyap/code/algodev/resources]
 <ResManager:DatasetRM>[0] scan(FIRST:0) in Locator<EIA|'/datasets'> [/home/ilyap/code/algodev/resources]
 <ResManager:CollectionRM>[0] scan(FIRST:0) in Locator<EIA|'/collections'> [/home/ilyap/code/algodev/resources]


Resource classes must be defined (or imported) to be recognized and managed.

In [8]:
for i in range(2, 5):
    rmA.add_resource(rmA.model(f'a{i}'), over=True)
rmA.list()

[🗸a1 ⟨⟩, 🗸a2 ⟨⟩, 🗸a3 ⟨⟩, 🗸a4 ⟨⟩]

### Resource Discovery

Normally, resources are not created and registered manually in code, but defined in `yml` files and discovered semi-automatically.

To support this, a resource model class should be defined using `locatable` decorator allowing to specify default resorce locations.

In [9]:
# Prepare some temporal folde for experiments with resources 
loc = Path('/tmp/test_resources')
rmtree(loc); loc.mkdir()

In [10]:
@locatable(folders=loc, pattern='.*\.test\.yml')
class ResT(ResourceModel):
    desc: Optional[str]
    a: int
    b: float = 0.25

Create `r1.test.yml` file with configuration of _ResT_ kind of resource:

In [11]:
with (loc / "r1.test.yml").open('wt') as f:
    f.write('''# ResT::
    name: Res1
    desc: a test resource
    a: 10''')

In [12]:
rmT  = ResT._manager
print(f"Discovered {ResT} resources: ", rmT.discover())
print(rmT, rmT.list(), sep='\n')

Discovered <class '__main__.ResT'> resources:  1
<ResManager:ResT>[1] scan(FIRST:1) in Locator<EIA> [/tmp/test_resources]
[🗸Res1 ⟨path:/tmp/test_resources/r1.test.yml⟩]


In [13]:
rmT.list(attr='cfg')

[{'cfg_file_info': {'path': PosixPath('/tmp/test_resources/r1.test.yml'),
   'folder': PosixPath('/tmp/test_resources'),
   'base_name': 'r1',
   'suffix': '.test.yml'},
  'name': 'Res1',
  'desc': 'a test resource',
  'a': 10,
  'b': 0.25}]

**Notice!**  
Discovery mechanism uses internally additional type of resources, which appear in the global `ModelsManager` list:

In [14]:
mm.list()

[<ResManager:ResA>[4] scan(FIRST:0) in Locator<EIA> [],
 <ResManager:ResB>[1] scan(FIRST:0) in Locator<EIA> [],
 <ResManager:DataSourceRM>[0] scan(FIRST:0) in Locator<EIA>$INU_DATASETS=/home/ilyap/data/depth [],
 <ResManager:SchemeRM>[0] scan(FIRST:0) in Locator<EIA|'/schemes'> [/home/ilyap/code/algodev/resources],
 <ResManager:DatasetRM>[0] scan(FIRST:0) in Locator<EIA|'/datasets'> [/home/ilyap/code/algodev/resources],
 <ResManager:CollectionRM>[0] scan(FIRST:0) in Locator<EIA|'/collections'> [/home/ilyap/code/algodev/resources],
 <ResManager:ResT>[1] scan(FIRST:1) in Locator<EIA> [/tmp/test_resources]]

Resource discovery may be requiested either from the `ModelsManager` or specific resource class. 

Additional resource types are registed when the corresponding resource models classes are being imported.

To discover all the registered resource types in their default locations:

Specific resources can be discovered corresponding resources can be discovered by specifically resuestin ther

In [15]:
print("Discovered of all resource types:", mm.discover())
mm

Discovered of all resource types: 27


<ResManager:ResA>[4] scan(FIRST:0) in Locator<EIA> []
<ResManager:ResB>[1] scan(FIRST:0) in Locator<EIA> []
<ResManager:DataSourceRM>[5] scan(FIRST:1) in Locator<EIA>$INU_DATASETS=/home/ilyap/data/depth []
<ResManager:SchemeRM>[21] scan(FIRST:1) in Locator<EIA|'/schemes'> [/home/ilyap/code/algodev/resources]
<ResManager:DatasetRM>[1] scan(FIRST:1) in Locator<EIA|'/datasets'> [/home/ilyap/code/algodev/resources]
<ResManager:CollectionRM>[0] scan(FIRST:1) in Locator<EIA|'/collections'> [/home/ilyap/code/algodev/resources]
<ResManager:ResT>[1] scan(FIRST:1) in Locator<EIA> [/tmp/test_resources]

In [16]:
mm['ResA'].list(columns=['name', 'path'])

,name,path
0,a1,None
1,a2,None
2,a3,None
3,a4,None


In [21]:
mm['ResA']['a1']

<ResAModel> {name: a1, description: first resource}